# Lab 1E: APIM Failover with Backend Pool & Circuit Breaker

Demonstrate **automatic, transparent failover** from one Azure OpenAI region to another
using APIM backend pools, circuit breakers, and retry policies.

## Architecture

```
Client  ──►  APIM (openai-failover-pool)
                 ├── Priority 1: openai (eastus2, 1K TPM) ── circuit breaker
                 └── Priority 2: openai-swedencentral (30K TPM) ── circuit breaker
```

## How It Works

| Component | Purpose |
|-----------|--------|
| **Backend Pool** | Groups backends with priority-based routing |
| **Circuit Breaker** | Trips on 429 → stops routing to that backend for `tripDuration` |
| **`acceptRetryAfter`** | Respects `Retry-After` header from Azure OpenAI for precise recovery |
| **Retry Policy** | Transparently retries the request on the next pool member (client sees 200, not 429) |

## Key Design

- **Primary** (eastus2): Intentionally set to **1K TPM** to trigger 429 quickly
- **Fallback** (swedencentral): Higher TPM to absorb overflow
- Both backends deploy the **same model name** (`gpt-4.1`) so URL path is identical

> ⚠️ **Prerequisite**: Complete **Lab 1A** first to deploy the Landing Zone,
> or deploy the included `main.bicep` standalone.

In [17]:
import sys
sys.path.insert(0, "../..")
from secure_print import install
install()

secure_print: Azure resource masking enabled


## Step 1: Load Environment

In [18]:
import os, subprocess, json
from pathlib import Path

env_file = Path("../../.env")
if env_file.exists():
    with open(env_file) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                os.environ[key] = value

APIM_URL = os.environ.get('APIM_URL', '')
APIM_KEY = os.environ.get('APIM_KEY', '')
RG = "lab1a-foundry-lz-hub"

# If running standalone, these will be set after deployment
print(f"APIM URL:  {APIM_URL}")
print(f"APIM Key:  {APIM_KEY[:4]}... (hidden)")

APIM URL:  https://fou***.azure-api.net/openai
APIM Key:  9c8f... (hidden)


## Step 2: Review Current APIM Configuration

Check backends, backend pool, and the chat operation policy.

In [19]:
import subprocess, json

APIM_NAME = APIM_URL.split("//")[1].split(".")[0] if "//" in APIM_URL else "foundry-apim-cjbd3s"
SUB = subprocess.run('az account show --query id -o tsv', shell=True,
                     capture_output=True, text=True).stdout.strip()

BASE = f"https://management.azure.com/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}"
API_VER = "api-version=2024-06-01-preview"

print("=" * 60)
print("BACKENDS")
print("=" * 60)
r = subprocess.run(
    f'az rest --method GET --url "{BASE}/backends?{API_VER}" '
    f'--query "value[].{{name:name, url:properties.url, type:properties.type, circuitBreaker:properties.circuitBreaker.rules[0].name}}" '
    f'-o table',
    shell=True, capture_output=True, text=True
)
print(r.stdout)

print("=" * 60)
print("API OPERATIONS")
print("=" * 60)
r = subprocess.run(
    f'az rest --method GET --url "{BASE}/apis/openai/operations?{API_VER}" '
    f'--query "value[].{{name:name, method:properties.method, urlTemplate:properties.urlTemplate}}" '
    f'-o table',
    shell=True, capture_output=True, text=True
)
print(r.stdout)

BACKENDS
Name                   Url                                                                           CircuitBreaker
---------------------  ----------------------------------------------------------------------------  -----------------
openai                 https://fou***.cognitiveservices.azure.com/openai                 breakOnThrottling
openai-failover-pool
openai-norwayeast      https://fou***.cognitiveservices.azure.com/openai
openai-southcentralus  https://fou***.cognitiveservices.azure.com/openai
openai-swedencentral   https://fou***.cognitiveservices.azure.com/openai   breakOnThrottling

API OPERATIONS
Name                  Method    UrlTemplate
--------------------  --------  ----------------------------------------------------
chat                  POST      /deployments/{deployment-id}/chat/completions
chat-norwayeast       POST      /deployments/o3-deep-research/chat/completions
chat-southcentralus   POST      /deployments/gpt-4.1-southcentralus/chat/completions
ch

## Step 3: Configure Failover (if not already deployed via Bicep)

This step applies the three key changes:
1. **Circuit breakers** on both backends (trip on 429)
2. **Backend pool** with priority-based routing
3. **Retry policy** on the chat operation for transparent failover

In [20]:
import json, subprocess

def apim_rest(method, path, body=None):
    """Helper to call APIM REST API."""
    url = f"{BASE}/{path}?{API_VER}"
    cmd = f'az rest --method {method} --url "{url}"'
    if body:
        # Write body to a temp file to avoid shell escaping issues on Windows
        import tempfile, os
        tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
        json.dump(body, tmp)
        tmp.close()
        cmd += f' --body "@{tmp.name}"'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if body:
        os.unlink(tmp.name)
    if r.returncode != 0:
        if 'Not a json response' in r.stderr:
            return r.stdout
        print(f"  ERROR: {r.stderr[:200]}")
        return None
    return json.loads(r.stdout) if r.stdout.strip().startswith('{') else r.stdout

In [21]:
# --- 3a: Get current backend URLs ---
backends_resp = apim_rest('GET', 'backends')
backend_urls = {b['name']: b['properties']['url'] for b in backends_resp['value']}

primary_url = backend_urls.get('openai', '')
fallback_url = backend_urls.get('openai-swedencentral', '')
print(f"Primary backend URL:  {primary_url}")
print(f"Fallback backend URL: {fallback_url}")

# Circuit breaker config (same for both backends)
cb_config = {
    "rules": [{
        "name": "breakOnThrottling",
        "failureCondition": {
            "count": 1,
            "interval": "PT10S",
            "statusCodeRanges": [{"min": 429, "max": 429}]
        },
        "tripDuration": "PT10S",
        "acceptRetryAfter": True
    }]
}

# --- 3b: Update primary backend with circuit breaker ---
print("\n🔧 Updating primary backend with circuit breaker...")
apim_rest('PUT', 'backends/openai', {
    "properties": {
        "url": primary_url,
        "protocol": "http",
        "circuitBreaker": cb_config
    }
})
print("✅ Primary backend: circuit breaker ON (trip on 429, 10s duration, acceptRetryAfter)")

# --- 3c: Update fallback backend with circuit breaker ---
print("\n🔧 Updating fallback backend with circuit breaker...")
apim_rest('PUT', 'backends/openai-swedencentral', {
    "properties": {
        "url": fallback_url,
        "protocol": "http",
        "description": "Sweden Central hub for GPT-4.1 failover",
        "circuitBreaker": cb_config
    }
})
print("✅ Fallback backend: circuit breaker ON")

Primary backend URL:  https://fou***.cognitiveservices.azure.com/openai
Fallback backend URL: https://fou***.cognitiveservices.azure.com/openai

🔧 Updating primary backend with circuit breaker...
✅ Primary backend: circuit breaker ON (trip on 429, 10s duration, acceptRetryAfter)

🔧 Updating fallback backend with circuit breaker...
✅ Fallback backend: circuit breaker ON


In [22]:
# --- 3d: Create backend pool with priority-based failover ---
print("🔧 Creating backend pool (openai-failover-pool)...")
primary_id = f"/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/backends/openai"
fallback_id = f"/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/backends/openai-swedencentral"

apim_rest('PUT', 'backends/openai-failover-pool', {
    "properties": {
        "type": "Pool",
        "pool": {
            "services": [
                {"id": primary_id,  "priority": 1, "weight": 1},
                {"id": fallback_id, "priority": 2, "weight": 1},
            ]
        }
    }
})
print("✅ Backend pool created:")
print("   Priority 1 → openai (eastus2)")
print("   Priority 2 → openai-swedencentral")

🔧 Creating backend pool (openai-failover-pool)...
✅ Backend pool created:
   Priority 1 → openai (eastus2)
   Priority 2 → openai-swedencentral


In [23]:
# --- 3e: Set chat operation policy to use the failover pool + retry ---
FAILOVER_POLICY = """<policies>
  <inbound>
    <base />
    <set-backend-service backend-id="openai-failover-pool" />
    <authentication-managed-identity resource="https://cognitiveservices.azure.com"
        output-token-variable-name="msi-access-token" ignore-error="false" />
    <set-header name="Authorization" exists-action="override">
      <value>@("Bearer " + (string)context.Variables["msi-access-token"])</value>
    </set-header>
  </inbound>
  <backend>
    <retry condition="@(context.Response.StatusCode == 429)"
           count="3" interval="0" first-fast-retry="true">
      <forward-request buffer-request-body="true" />
    </retry>
  </backend>
  <outbound>
    <base />
  </outbound>
</policies>"""

print("🔧 Applying failover policy to 'chat' operation...")
apim_rest('PUT', 'apis/openai/operations/chat/policies/policy', {
    "properties": {
        "format": "xml",
        "value": FAILOVER_POLICY
    }
})
print("✅ Chat operation policy updated with:")
print("   • set-backend-service → openai-failover-pool")
print("   • retry on 429 (count=3, first-fast-retry=true)")
print("   • managed-identity auth to Cognitive Services")

🔧 Applying failover policy to 'chat' operation...
✅ Chat operation policy updated with:
   • set-backend-service → openai-failover-pool
   • retry on 429 (count=3, first-fast-retry=true)
   • managed-identity auth to Cognitive Services


## Step 4: Verify Model Deployments

Both hubs must have the **same deployment name** (`gpt-4.1`) for transparent failover.  
The primary should have **low TPM** (1K) to trigger 429 easily.

In [24]:
# Extract hub names from backend URLs
primary_hub = primary_url.split("//")[1].split(".")[0]
fallback_hub = fallback_url.split("//")[1].split(".")[0]

print(f"Primary hub:  {primary_hub}")
print(f"Fallback hub: {fallback_hub}")

for hub in [primary_hub, fallback_hub]:
    print(f"\n{'='*50}")
    print(f"Deployments on {hub}")
    print(f"{'='*50}")
    r = subprocess.run(
        f'az rest --method GET '
        f'--url "https://management.azure.com/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.CognitiveServices/accounts/{hub}/deployments?api-version=2024-10-01" '
        f'--query "value[].{{name:name, model:properties.model.name, sku:sku.name, capacity:sku.capacity}}" '
        f'-o table',
        shell=True, capture_output=True, text=True
    )
    print(r.stdout)

Primary hub:  foundry-hub-cjbd3s
Fallback hub: foundry-hub-swedencentral-cjbd3s

Deployments on foundry-hub-cjbd3s
Name                    Model                   Sku             Capacity
----------------------  ----------------------  --------------  ----------
text-embedding-3-large  text-embedding-3-large  Standard        50
gpt-4.1                 gpt-4.1                 GlobalStandard  1


Deployments on foundry-hub-swedencentral-cjbd3s
Name     Model    Sku             Capacity
-------  -------  --------------  ----------
gpt-4.1  gpt-4.1  GlobalStandard  190



## Step 5: Test Failover via REST

Send rapid requests through APIM. With 1K TPM on the primary backend:
1. First call → **East US 2** (primary, within quota)
2. Primary returns 429 → circuit breaker **trips** → retry policy sends to fallback
3. Subsequent calls → **Sweden Central** (pool routes to priority 2)

The client sees **200 OK for every call** — the 429 is handled transparently by APIM.

In [25]:
import requests
import time
from collections import Counter

APIM_GATEWAY = APIM_URL.rsplit('/openai', 1)[0]  # e.g. https://foundry-apim-xxx.azure-api.net
CHAT_URL = f"{APIM_GATEWAY}/openai/deployments/gpt-4.1/chat/completions?api-version=2024-10-21"
HEADERS = {"api-key": APIM_KEY, "Content-Type": "application/json"}
PAYLOAD = {"messages": [{"role": "user", "content": "Say just 'ok'"}], "max_tokens": 5}

N_CALLS = 10
results = []

print(f"Sending {N_CALLS} rapid calls to: {CHAT_URL}")
print(f"Primary (eastus2) has LOW TPM → should trigger 429 → failover to swedencentral\n")

for i in range(1, N_CALLS + 1):
    try:
        r = requests.post(CHAT_URL, headers=HEADERS, json=PAYLOAD, timeout=30)
        region = r.headers.get('x-ms-region', 'unknown')
        status = r.status_code
        results.append({"call": i, "status": status, "region": region})
        symbol = "✅" if status == 200 else "⚠️"
        print(f"  Call {i:02d} | {symbol} {status} | Region: {region}")
    except Exception as e:
        results.append({"call": i, "status": "ERR", "region": "error"})
        print(f"  Call {i:02d} | ❌ ERROR: {str(e)[:80]}")
    time.sleep(0.3)

Sending 10 rapid calls to: https://fou***.azure-api.net/openai/deployments/gpt-4.1/chat/completions?api-version=2024-10-21
Primary (eastus2) has LOW TPM → should trigger 429 → failover to swedencentral

  Call 01 | ✅ 200 | Region: East US 2
  Call 02 | ✅ 200 | Region: Sweden Central
  Call 03 | ✅ 200 | Region: Sweden Central
  Call 04 | ✅ 200 | Region: Sweden Central
  Call 05 | ✅ 200 | Region: Sweden Central
  Call 06 | ✅ 200 | Region: Sweden Central
  Call 07 | ✅ 200 | Region: Sweden Central
  Call 08 | ✅ 200 | Region: Sweden Central
  Call 09 | ✅ 200 | Region: Sweden Central
  Call 10 | ✅ 200 | Region: Sweden Central


In [26]:
# Analyze results
print("\n" + "=" * 60)
print("FAILOVER TEST RESULTS")
print("=" * 60)

regions = [r['region'] for r in results if r['status'] == 200]
region_counts = Counter(regions)

print(f"\nTotal calls:    {len(results)}")
print(f"Successful:     {sum(1 for r in results if r['status'] == 200)}")
print(f"Failed (429):   {sum(1 for r in results if r['status'] == 429)}")

print(f"\nRegion distribution:")
for region, count in region_counts.most_common():
    pct = count / len(regions) * 100 if regions else 0
    bar = '█' * int(pct / 5)
    print(f"  {region:20s}  {count:3d} ({pct:5.1f}%)  {bar}")

east_seen = any('East US 2' in r for r in regions)
sweden_seen = any('Sweden' in r for r in regions)

print(f"\nVerification:")
print(f"  East US 2 served requests:      {east_seen}")
print(f"  Sweden Central served requests:  {sweden_seen}")

if east_seen and sweden_seen:
    print("\n✅ FAILOVER VERIFIED: Traffic moved from East US 2 → Sweden Central")
    print("   The 429 was handled transparently by APIM — client saw 200 for all calls.")
elif sweden_seen:
    print("\n✅ All traffic routed to Sweden Central (primary circuit breaker still tripped from previous test)")
elif east_seen:
    print("\nℹ️  All traffic stayed on East US 2 — primary wasn't throttled. Try sending more calls.")
else:
    print("\n⚠️  Unexpected results — check APIM configuration.")


FAILOVER TEST RESULTS

Total calls:    10
Successful:     10
Failed (429):   0

Region distribution:
  Sweden Central          9 ( 90.0%)  ██████████████████
  East US 2               1 ( 10.0%)  ██

Verification:
  East US 2 served requests:      True
  Sweden Central served requests:  True

✅ FAILOVER VERIFIED: Traffic moved from East US 2 → Sweden Central
   The 429 was handled transparently by APIM — client saw 200 for all calls.


## Step 6: Test with curl (copy/paste)

You can also test manually with curl:

In [27]:
print("Run this curl command to test failover:\n")
print(f"""curl -s -w "\\n%{{http_code}} | Region: %{{header:x-ms-region}}" \\
  -X POST "{CHAT_URL}" \\
  -H "api-key: {APIM_KEY}" \\
  -H "Content-Type: application/json" \\
  -d '{json.dumps(PAYLOAD)}'""")

print("\n" + "-" * 60)
print("Or run it 10 times rapidly to trigger failover:")
print(f"""for i in $(seq 1 10); do
  curl -s -w "Call $i | %{{http_code}} | %{{header:x-ms-region}}\\n" -o /dev/null \\
    -X POST "{CHAT_URL}" \\
    -H "api-key: {APIM_KEY}" \\
    -H "Content-Type: application/json" \\
    -d '{json.dumps(PAYLOAD)}'
done""")

Run this curl command to test failover:

curl -s -w "\n%{http_code} | Region: %{header:x-ms-region}" \
  -X POST "https://fou***.azure-api.net/openai/deployments/gpt-4.1/chat/completions?api-version=2024-10-21" \
  -H "api-key: 9c8f***a5d1" \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Say just 'ok'"}], "max_tokens": 5}'

------------------------------------------------------------
Or run it 10 times rapidly to trigger failover:
for i in $(seq 1 10); do
  curl -s -w "Call $i | %{http_code} | %{header:x-ms-region}\n" -o /dev/null \
    -X POST "https://fou***.azure-api.net/openai/deployments/gpt-4.1/chat/completions?api-version=2024-10-21" \
    -H "api-key: 9c8f***a5d1" \
    -H "Content-Type: application/json" \
    -d '{"messages": [{"role": "user", "content": "Say just 'ok'"}], "max_tokens": 5}'
done


## How the APIM Failover Policy Works

The complete policy applied to the chat operation:

```xml
<policies>
  <inbound>
    <base />
    <!-- Route to the failover pool instead of a single backend -->
    <set-backend-service backend-id="openai-failover-pool" />
    <!-- Managed identity auth (no API keys needed for backend) -->
    <authentication-managed-identity resource="https://cognitiveservices.azure.com"
        output-token-variable-name="msi-access-token" ignore-error="false" />
    <set-header name="Authorization" exists-action="override">
      <value>@("Bearer " + (string)context.Variables["msi-access-token"])</value>
    </set-header>
  </inbound>
  <backend>
    <!-- Retry transparently on 429: circuit breaker trips the primary,
         retry sends to next pool member (swedencentral) -->
    <retry condition="@(context.Response.StatusCode == 429)"
           count="3" interval="0" first-fast-retry="true">
      <forward-request buffer-request-body="true" />
    </retry>
  </backend>
  <outbound>
    <base />
  </outbound>
</policies>
```

### Key Components

| Component | Config | Purpose |
|-----------|--------|--------|
| Circuit Breaker | `count: 1, interval: PT10S, statusCode: 429` | Trip after 1 failure in 10s window |
| Circuit Breaker | `tripDuration: PT10S, acceptRetryAfter: true` | Stay open 10s or per Retry-After header |
| Backend Pool | `priority: 1 (eastus2), priority: 2 (swedencentral)` | Route to primary first, fallback on trip |
| Retry Policy | `count: 3, first-fast-retry: true` | Retry 429 immediately on next pool member |

### Sequence Diagram

```
Client ──► APIM ──► Pool routes to eastus2 (priority 1)
                       └── 429 returned
                       └── Circuit breaker TRIPS on eastus2
                       └── Retry policy kicks in
           APIM ──► Pool routes to swedencentral (priority 2)
                       └── 200 OK
       ◄── 200 OK returned to client (transparent!)
```

## Cleanup (Optional)

Restore the primary backend capacity back to normal.

In [28]:
# # Restore primary capacity to 30K TPM
# import subprocess, json, tempfile, os
# body = {"sku": {"name": "GlobalStandard", "capacity": 30}}
# tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
# json.dump(body, tmp); tmp.close()
# r = subprocess.run(
#     f'az rest --method PATCH '
#     f'--url "https://management.azure.com/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.CognitiveServices/accounts/{primary_hub}/deployments/gpt-4.1?api-version=2024-10-01" '
#     f'--body "@{tmp.name}" --query "{{name:name, sku:sku}}" -o json',
#     shell=True, capture_output=True, text=True
# )
# os.unlink(tmp.name)
# print(r.stdout)
# print("✅ Primary capacity restored to 30K TPM")